# 📊 ĐÁNH GIÁ, TRỰC QUAN HÓA TOÁN HỌC & SO SÁNH MÔ HÌNH (BENCHMARK)
> Notebook này cung cấp công cụ trực quan hóa toàn diện:
1. **Phân tích toán học:** Vẽ dạng sóng $f(x)$ và đạo hàm Gradient $\frac{df(x)}{dx}$ của các hàm kích hoạt (PReLU, GELU, SiLU, SoftClamp, SmoothPReLU).
2. **So sánh mô hình:** Vẽ đường cong hội tụ Val PSNR qua 40 Epochs, biểu đồ so sánh tác động của khối **ECA Attention** và bảng chỉ số $\Delta$PSNR.

In [ ]:
# 1. KẾT NỐI GOOGLE DRIVE & DI CHUYỂN VÀO THƯ MỤC DỰ ÁN
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/COMPUTER SCIENCE/NAM3_HK3_(2025-2026)/CT282_Deep Learning/PROJECT/RIFE-MinhTri/RIFE-Project"
os.chdir(PROJECT_DIR)
print(f"✅ Đang ở thư mục dự án: {os.getcwd()}")

---
## 🎨 PHẦN 1: SO SÁNH CÁC HÀM KÍCH HOẠT & ĐẠO HÀM GRADIENT

In [ ]:
# 2. VẼ SO SÁNH DẠNG SÓNG F(X) VÀ ĐẠO HÀM DF(X)/DX CỦA CÁC HÀM KÍCH HOẠT
from visualize import plot_activations_and_gradients

plot_activations_and_gradients(save_path='demo/activations_comparison.png')

---
## 📊 PHẦN 2: SO SÁNH HIỆU NĂNG CÁC MÔ HÌNH VÀ KHỐI ECA ATTENTION

In [ ]:
# 3. TỰ ĐỘNG QUÉT TRAINED_MODEL/ VÀ VẼ BIỂU ĐỒ HỘI TỤ PSNR & BEST PSNR
from visualize import plot_model_comparisons
from IPython.display import display

df_summary = plot_model_comparisons(models_dir='trained_model', save_path='demo/benchmark_models_comparison.png')

if df_summary is not None:
    print("\n📋 BẢNG TỔNG KẾT SO SÁNH CHỈ SỐ ΔPSNR SO VỚI BASELINE GỐC:")
    display(df_summary)

---
## 🔍 PHẦN 3: PHÂN TÍCH CHUYÊN SÂU TÁC ĐỘNG CỦA KHỐI ECA ATTENTION

In [ ]:
# 4. PHÂN TÍCH TÁC ĐỘNG RIÊNG LẺ CỦA ECA TRÊN TỪNG HÀM KÍCH HOẠT (BEFORE vs AFTER ECA)
import matplotlib.pyplot as plt
import pandas as pd
import os
import json

models_dir = 'trained_model'
acts = ['prelu', 'gelu', 'silu', 'smooth_prelu', 'optimized_smooth_prelu', 'soft_clamp_relu', 'soft_clamp_silu']
eca_comparison = []

for act in acts:
    base_file = os.path.join(models_dir, f'baseline_{act}', 'experiment_results.json')
    eca_file = os.path.join(models_dir, f'modify_eca_{act}', 'experiment_results.json')
    
    base_psnr = None
    eca_psnr = None
    
    if os.path.exists(base_file):
        with open(base_file) as f:
            d = json.load(f)
            base_psnr = max([x['val_psnr'] for x in d]) if d else None
            
    if os.path.exists(eca_file):
        with open(eca_file) as f:
            d = json.load(f)
            eca_psnr = max([x['val_psnr'] for x in d]) if d else None
            
    if base_psnr is not None or eca_psnr is not None:
        delta_eca = (eca_psnr - base_psnr) if (eca_psnr and base_psnr) else None
        eca_comparison.append({
            'Hàm Kích Hoạt': act.upper(),
            'Baseline (Không ECA)': f'{base_psnr:.2f} dB' if base_psnr else 'Chưa train',
            'Modify (+ Khối ECA)': f'{eca_psnr:.2f} dB' if eca_psnr else 'Chưa train',
            'Độ Tăng Trưởng (+ ΔECA)': f'{delta_eca:+.2f} dB' if delta_eca is not None else 'N/A'
        })

if eca_comparison:
    df_eca = pd.DataFrame(eca_comparison)
    print('✨ BẢNG SO SÁNH TÁC ĐỘNG TĂNG TRƯỞNG KHI THÊM ECA ATTENTION:')
    display(df_eca)
else:
    print('⚠️ Hãy train cả mô hình baseline_* và modify_eca_* để xem bảng so sánh tăng trưởng ECA.')